# Empirical Evaluation of DQN Variants on Atari 2600
### TEK-3601 / AUT-2802 Machine Vision — UiT The Arctic University of Norway
**Project**: Q-Value Overestimation and Action-Value Decomposition across 4 Game Categories

---
### 1. Research Objectives
* **RQ1 (Value Overestimation)**: Does Double Q-learning eliminate positive Q-value overestimation bias compared to Vanilla DQN across reactive vs. strategic games?
* **RQ2 (Action-Value Decomposition)**: Does the Dueling architecture improve sample efficiency when multiple discrete actions yield identical environmental states?

### 2. The 4 Evaluated Atari Games & Categories
1. **Pong** (`ALE/Pong-v5`): Reflexive / Reactive 2D Ball & Paddle (Small 3-action effective space, dense immediate feedback)
2. **Space Invaders** (`ALE/SpaceInvaders-v5`): Fixed-Screen Defensive Tactical Shooter (Spatial defense, delayed risk)
3. **Ms. Pacman** (`ALE/MsPacman-v5`): Maze Navigation & Dense Spatial Foraging (Corridor action redundancy, dense rewards)
4. **Seaquest** (`ALE/Seaquest-v5`): Multi-Objective Resource Management & Delayed Horizon (18 actions, oxygen gauge, delayed survival reward)


In [ ]:
# Install required dependencies for Google Colab / local environment
!pip install -q "gymnasium[atari]" ale-py torch torchvision torchaudio matplotlib pandas tqdm


In [ ]:
import os, random, copy, time
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')


### 3. Human and Random Baselines for Normalization
Standard Atari-57 scores from DeepMind Nature (Mnih et al., 2015) and Google Dopamine.
$$HNS = \frac{\text{Score} - \text{Random}}{\text{Human} - \text{Random}}$$


In [ ]:
from src.normalization import ATARI_RANDOM_SCORES, ATARI_HUMAN_SCORES, compute_human_normalized_score

eval_games = ['pong', 'spaceinvaders', 'mspacman', 'seaquest']
for g in eval_games:
    r = ATARI_RANDOM_SCORES[g]
    h = ATARI_HUMAN_SCORES[g]
    print(f'{g:15s} | Random Baseline: {r:8.1f} | Human Baseline: {h:8.1f}')


### 4. Atari Visual Preprocessing Wrappers (Nature CNN Pipeline)
- Grayscale conversion + 84x84 bilinear resize
- Max-pooling over 2 consecutive frames with 4-frame skipping
- 4-frame temporal stacking
- Episodic life termination during training
- Reward clipping {-1, 0, +1}


In [ ]:
from src.wrappers import make_atari_env

# Visualize the 4 selected games
games = [('pong', 'ALE/Pong-v5'), ('spaceinvaders', 'ALE/SpaceInvaders-v5'),
         ('mspacman', 'ALE/MsPacman-v5'), ('seaquest', 'ALE/Seaquest-v5')]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, env_id) in zip(axes, games):
    env = gym.make(env_id, render_mode='rgb_array')
    env.reset(seed=42)
    for _ in range(40):
        env.step(env.action_space.sample())
    ax.imshow(env.render())
    ax.set_title(f'{name.upper()}\n({env.action_space.n} Actions)', fontweight='bold')
    ax.axis('off')
    env.close()
plt.suptitle('The 4 Evaluated Atari 2600 Environments', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 5. Neural Network Architectures
We implement **Nature CNN** as the common visual backbone, connected to:
1. Standard Q-Head (for Vanilla DQN & Double DQN)
2. Decoupled Value $V(s)$ and Advantage $A(s, a)$ heads (for Dueling DQN) with mean-centering constraint:
$$Q(s, a) = V(s) + \left(A(s, a) - \frac{1}{|\mathcal{A}|}\sum_{a'} A(s, a')\right)$$


In [ ]:
from src.models import StandardDQN, DuelingDQN, build_network
from src.replay_buffer import ReplayBuffer
from src.agent import DQNAgent

# Verify model parameter count
std_model = StandardDQN(in_channels=4, num_actions=6)
duel_model = DuelingDQN(in_channels=4, num_actions=6)

print(f'Standard DQN Parameters: {sum(p.numel() for p in std_model.parameters()):,}')
print(f'Dueling DQN Parameters:  {sum(p.numel() for p in duel_model.parameters()):,}')


### 6. Training Pipeline (Multi-Seed & Multi-Game)
Execute training across algorithm variants and seeds with holdout Q-value overestimation tracking.


In [ ]:
from src.train import train

# Example: Run 500k timesteps training on Pong
# Uncomment below to execute training in Colab:
# metrics = train(
#     algo='dueling_dqn',
#     game_key='pong',
#     seed=1,
#     total_timesteps=500_000,
#     eval_frequency=25_000,
#     eval_episodes=30,
#     output_dir='runs',
#     device=device
# )


### 7. Evaluation & Scientific Visualization
Plotting learning curves, Q-bias curves, and Human-Normalized Score bar charts.


In [ ]:
from src.visualize import load_runs_data, plot_learning_curves, plot_q_value_overestimation, plot_human_normalized_comparison

# Load experimental results
runs_data = load_runs_data('runs')
if runs_data:
    plot_learning_curves(runs_data, 'learning_curves.png')
    plot_q_value_overestimation(runs_data, 'q_overestimation_curves.png')
    print('Plots generated successfully!')
else:
    print('No runs found in runs/ directory. Train agents first to generate plots.')
